# Sesión 01 — Fundamentos de Python

**Bloque:** 1 · Fundamentos

## Objetivos

Al terminar esta sesión deberías ser capaz de:

- Elegir entre una lista y un diccionario justificando la decisión con cómo vas a buscar
  los datos, no con cuál te suena mejor.
- Escribir una función que haga **una** cosa, devuelva un valor y no imprima nada.
- Aislar un dato corrupto con `try-except` en vez de dejar que reviente el programa entero,
  y decidir qué hacer con él.
- Sustituir un bucle que construye una lista por una comprensión, y reconocer cuándo la
  comprensión empeora la lectura.
- Modelar con una clase algo que tiene datos **y** reglas propias, y explicar por qué ahí un
  diccionario se queda corto.
- Explicar qué se pierde al descartar los registros defectuosos de un fichero.

## Qué necesitas de antes

Nada. Es la primera sesión. Hace falta Python instalado y saber abrir un notebook; todo lo
demás se construye aquí.

### Cómo funciona este material

Antes de empezar, tres cosas sobre el cuaderno que tienes delante, porque condicionan cómo
sacarle partido:

- **Las celdas de demostración se ejecutan en clase y están resueltas.** No hay que
  copiarlas. Son para mirar qué pasa.
- **Las celdas de ejercicio están vacías, con una pista en forma de comentario.** La pista
  orienta pero no resuelve, y eso es a propósito.
- **Las celdas de "Mis reflexiones" son la parte que de verdad se evalúa.** Este curso no se
  examina de memoria: se defiende en voz alta. Puedes usar una IA para escribir el código
  —se asume que lo harás— pero la pregunta de la defensa no va a ser *qué* hiciste, va a ser
  *por qué eso y no otra cosa*. Esa respuesta no la tiene la IA: depende de un contexto que
  solo conoces tú. Escribe las reflexiones según avanzas, no al final.

## 1. Listas y diccionarios: la decisión es cómo vas a buscar

Casi todo lo que vas a hacer en este curso empieza con una de estas dos estructuras, y la
elección no es de estilo. Depende de una sola pregunta: **¿cómo vas a acceder a los datos
después?**

- Una **lista** es una secuencia ordenada: `ventas[0]`, `ventas[1]`. Tiene sentido cuando el
  orden significa algo o cuando vas a recorrerla entera. Buscar dentro de ella por contenido
  obliga a mirar elemento por elemento.
- Un **diccionario** es un conjunto de pares clave-valor: `cliente['nombre']`. Tiene sentido
  cuando vas a buscar por una etiqueta, y encuentra lo que busca **sin recorrer nada**, dé
  igual que haya diez claves o diez millones.

La combinación que más vas a usar es la de los dos juntos: **una lista de diccionarios**.
Cada diccionario es un registro con sus campos, y la lista es la colección de registros. Eso
es, literalmente, una tabla. Cuando en la sesión 02 aparezca el `DataFrame` de pandas,
piénsalo como esta misma idea con esteroides.

**Cuándo se usa una lista:** hay orden, o hay que recorrerlo todo.
**Cuándo NO:** cuando la pregunta es "¿cuánto lleva acumulado el producto X?" — ahí un
diccionario te lo da directo.
**Qué asume el diccionario:** que las claves son únicas. Si asignas dos veces la misma
clave, la segunda pisa a la primera sin decir nada.

### Demostración guiada — las dos estructuras y su combinación

La ejecutamos juntos. Fíjate en el patrón de `.get()` del final: es el que se usa para
acumular, y aparece en los tres ejercicios.

In [7]:
# --- 1. Lista: el orden importa, se accede por posición ------------------
temperaturas = [15.5, 17.2, 16.8, 18.1]
print("Toda la lista :", temperaturas)
print("La primera    :", temperaturas[0])
print("La última     :", temperaturas[-1])
print("Cuántas hay   :", len(temperaturas))

# --- 2. Diccionario: se accede por nombre --------------------------------
cliente = {'nombre': 'Ana Torres', 'ciudad': 'Valencia', 'pedidos': 3}
print("\nEl cliente    :", cliente)
print("Solo el nombre:", cliente['nombre'])

# Con corchetes, una clave que no existe revienta. Con .get() devuelve None
# (o lo que le digas), y eso evita la mitad de los try-except de un programa.
print("Email con .get():", cliente.get('email'))
print("Email con defecto:", cliente.get('email', 'sin email'))

# --- 3. La combinación: una lista de diccionarios es una tabla -----------
pedidos = [
    {'cliente': 'Ana', 'importe': 120.0},
    {'cliente': 'Luis', 'importe': 45.5},
    {'cliente': 'Ana', 'importe': 80.0},
]
print("\nPedidos:")
for p in pedidos:
    print(f"  {p['cliente']:6} {p['importe']:>8.2f} €")

# --- 4. El patrón de acumular: diccionario + .get() ----------------------
# "Cuánto lleva gastado cada cliente". La clave es el cliente, el valor se suma.
total_por_cliente = {}
for p in pedidos:
    total_por_cliente[p['cliente']] = total_por_cliente.get(p['cliente'], 0) + p['importe']

print("\nTotal por cliente:", total_por_cliente)

# Y para quedarte con el mayor, sin ordenar nada:
mejor = max(total_por_cliente, key=total_por_cliente.get)
print(f"El que más gasta: {mejor} ({total_por_cliente[mejor]:.2f} €)")

Toda la lista : [15.5, 17.2, 16.8, 18.1]
La primera    : 15.5
La última     : 18.1
Cuántas hay   : 4

El cliente    : {'nombre': 'Ana Torres', 'ciudad': 'Valencia', 'pedidos': 3}
Solo el nombre: Ana Torres
Email con .get(): None
Email con defecto: sin email

Pedidos:
  Ana      120.00 €
  Luis      45.50 €
  Ana       80.00 €

Total por cliente: {'Ana': 200.0, 'Luis': 45.5}
El que más gasta: Ana (200.00 €)


**Qué se ve.** Los puntos 3 y 4 son los que hay que llevarse.

Una **lista de diccionarios** es la forma natural de guardar registros: cada uno con sus
campos, todos con la misma forma. Es exactamente lo que vas a construir en el ejercicio 1.

Y el **patrón de acumulación** del punto 4 merece leerse despacio, porque se repite tres
veces en esta sesión:

```python
acumulado[clave] = acumulado.get(clave, 0) + valor
```

Se lee así: *"coge lo que ya había para esta clave —y si no había nada, empieza en cero— y
súmale esto"*. Sin el `.get()` habría que preguntar antes si la clave existe, con un `if`,
en cada vuelta del bucle. El `max(..., key=...)` de la última línea es el remate: devuelve la
clave cuyo valor es el mayor, sin necesidad de ordenar el diccionario.

## 2. Funciones: una función, una responsabilidad

Una función es un trozo de código con nombre que recibe unos datos y **devuelve** un
resultado. Lo importante de esa frase es el verbo: devuelve, no imprime.

```python
def nombre(parametro):
    ...
    return resultado
```

La distinción entre `return` y `print` parece de principiante y no lo es. `print` **enseña**
un valor en pantalla y se acabó: nadie puede usarlo después. `return` **entrega** el valor a
quien llamó a la función, que puede guardarlo, sumarlo o pasárselo a otra. Una función que
solo imprime aparenta funcionar —ves el número— pero no sirve para construir nada encima.
Si la usas dentro de otra operación, lo que recibes es `None`.

La otra regla, que es la que separa el código que se puede mantener del que no: **una
función, una responsabilidad**. Una función que lee un fichero, calcula unas métricas y
además las imprime hace tres cosas, y no puedes reutilizar ninguna por separado ni probarla
sin ejecutarlas todas. Partirla en tres cuesta lo mismo de escribir y vale mucho más.

**Cuándo escribir una:** cuando algo se repite, o cuando un bloque necesita un comentario
para explicar qué hace — ese comentario suele ser el nombre de la función.
**Cuándo NO:** para una sola línea que ya se lee sola.

### Demostración guiada — devolver, no imprimir

Dos versiones de la misma idea. La primera es la trampa.

In [ ]:
# --- 1. La versión que ENGAÑA -------------------------------------------
def a_numero_mal(texto):
    print(float(texto.replace('.', '').replace(',', '.')))   # imprime, no devuelve


print("Llamando a la versión mala:")
resultado = a_numero_mal("1.234,50")
print("Lo que ha devuelto:", resultado)      # None: el valor se ha perdido

# --- 2. La versión que SIRVE --------------------------------------------
def a_numero(texto):
    """Convierte un número en formato español ('1.234,50') a float."""
    return float(texto.replace('.', '').replace(',', '.'))


precio = a_numero("1.234,50")
iva = a_numero("21,00")
print(f"\nAhora sí se puede operar: {precio} + {precio * iva / 100:.2f} de IVA")

# --- 3. Una responsabilidad cada una -------------------------------------
# Tres funciones pequeñas en vez de una que lo haga todo
def importe_linea(cantidad, precio):
    return cantidad * precio


def aplicar_descuento(importe, porcentaje):
    return importe * (1 - porcentaje / 100)


def formatear_euros(importe):
    return f"{importe:,.2f} €".replace(',', 'X').replace('.', ',').replace('X', '.')


bruto = importe_linea(3, 75.99)
neto = aplicar_descuento(bruto, 10)
print(f"\nBruto: {formatear_euros(bruto)}  ->  Con 10% dto: {formatear_euros(neto)}")

**Qué se ve.** La primera función imprime `1234.5` en pantalla y devuelve `None`. Es una
trampa real: en la consola *parece* que funciona, y solo se descubre cuando intentas usar el
resultado y Python se queja de que no puede sumar `None`.

Las tres funciones del punto 3 hacen cada una una cosa, y por eso se pueden encadenar:
`formatear_euros(aplicar_descuento(importe_linea(...)))`. Si mañana cambia la forma de
mostrar los euros, se toca una función y las otras dos ni se enteran. Escrito como una
función única de veinte líneas, ese cambio obliga a releerla entera para saber dónde meter
mano.

## 3. Datos sucios: `try-except`

Fuera de clase, los datos vienen mal. Una columna que debería ser un número trae la palabra
`ERROR`, una fila tiene tres campos en vez de cuatro, alguien escribió la fecha al revés. Y
en Python, un solo valor malo detiene el programa entero: `int("ERROR")` lanza un
`ValueError` y ahí se acaba el proceso, aunque las otras 999 filas estuvieran perfectas.

`try-except` sirve para **acotar el daño**:

```python
try:
    # lo que puede salir mal
except ValueError:
    # qué hacemos si sale mal
```

Dos reglas que valen para todo el curso:

**Captura el error concreto, no todos.** `except ValueError:` atrapa los fallos de
conversión. `except:` a secas atrapa *cualquier* cosa, incluidas tus erratas: si escribes
mal el nombre de una variable, el programa no fallará y tampoco hará nada, y te pasarás una
tarde buscándolo.

**Decidir qué hacer con lo malo es una decisión de negocio, no técnica.** Devolver `None` y
descartar la fila es lo más común, y es lo que haremos hoy. Pero descartar no es gratis, y
en el ejercicio 3 se ve por qué.

**Cuándo se usa:** cuando el fallo es *esperable* — datos externos, ficheros, red.
**Cuándo NO:** para tapar un error de tu propio código. Ahí el fallo es la información útil.

### Demostración guiada — acotar el daño

Tres versiones de lo mismo: la que revienta, la que se traga todo y la correcta.

In [5]:
lecturas = ["12.5", "13.1", "ERROR", "14.8"]

# --- 1. Sin protección: se para en el tercero ----------------------------
print("1. Sin try-except:")
try:
    total = sum(float(x) for x in lecturas)
    print("   total:", total)
except ValueError as e:
    print(f"   REVENTÓ y no procesó nada: {e}")

# --- 2. El except desnudo: funciona, y es mala idea ----------------------
def a_float_mal(texto):
    try:
        return float(texto)
    except:                      # se traga CUALQUIER error, también tus erratas
        return None


# --- 3. La versión correcta: error concreto ------------------------------
def a_float(texto):
    try:
        return float(texto)
    except ValueError:           # solo los fallos de conversión
        return None


print("\n3. Con try-except, procesando lo que se puede:")
validos = []
descartados = 0
for x in lecturas:
    valor = a_float(x)
    if valor is None:
        descartados += 1
        print(f"   descartada la lectura {x!r}")
    else:
        validos.append(valor)

print(f"   válidas: {len(validos)} | descartadas: {descartados}")
print(f"   total: {sum(validos):.1f}")

1. Sin try-except:
   REVENTÓ y no procesó nada: could not convert string to float: 'ERROR'

3. Con try-except, procesando lo que se puede:
   descartada la lectura 'ERROR'
   válidas: 3 | descartadas: 1
   total: 40.4


**Qué se ve.** Sin protección, un solo `"ERROR"` tira el cálculo completo y no se salva ni
una de las tres lecturas buenas. Con `try-except`, se procesan las tres y se deja constancia
de la que falló.

Fíjate en el detalle que separa el punto 2 del 3. Los dos funcionan con estos datos. La
diferencia aparece el día que escribes `float(texot)` con una errata: la versión del punto 2
devuelve `None` tan tranquila —se ha tragado un `NameError` creyendo que era un dato malo— y
tú te quedas mirando una lista vacía sin entender nada. La del punto 3 te enseña el error en
la cara, que es lo que quieres.

Y lo que no hay que pasar por alto: **el contador de descartados**. Un proceso que descarta
en silencio es un proceso en el que no se puede confiar. Saber cuántas filas se cayeron es
parte del resultado, no un detalle de depuración.

---
## Ejercicio 1 — Las ventas de la tienda

**Contexto.** Llevas una semana en una tienda de informática pequeña. El TPV vuelca las
ventas del día en un fichero de texto plano: una línea por venta, con
`producto,cantidad,precio_unitario`. El dueño te pide dos cosas para la reunión de mañana:
cuánto se ha facturado y cuál es su producto estrella. Te avisa de que el TPV "a veces
escribe cosas raras" y que del resto de la empresa nadie ha querido mirarlo.

**Tu tarea.**

1. **Crea una función `procesar_linea(linea)`:**
   - Recibe una línea de texto (por ejemplo `"Laptop,2,1250.50"`).
   - Separa los datos, convierte la cantidad a entero y el precio a flotante.
   - Si la línea tiene un error, lo maneja con `try-except` y devuelve `None`.
   - Si es correcta, devuelve un **diccionario** con las claves `producto`, `cantidad` y
     `precio`.
2. **Procesa todos los datos.** Crea una lista `ventas_procesadas`, recorre las líneas,
   llama a tu función por cada una y añade a la lista lo que no sea `None`. Lleva la cuenta
   de cuántas has descartado.
3. **Calcula las métricas:** los ingresos totales (suma de `cantidad * precio`) y el
   producto que más **unidades** ha vendido en total.

**Criterio de que lo has hecho bien:** tiene que descartarse exactamente una línea, y tienes
que saber cuál sin buscarla a mano. Si tus ingresos totales salen en cero o dan error, casi
seguro que estás sumando cadenas de texto en vez de números.

In [1]:
# No necesitas leer un archivo, usaremos esta variable como si fuera el contenido de uno.
datos_ventas_texto = """
Laptop,2,1250.50
Mouse,5,25.00
Teclado,3,75.99
Monitor,ERROR,450.00
Laptop,1,1200.00
Webcam,4,50.25
Mouse,10,22.50
Teclado,1,75.99
"""

In [10]:
# TODO: completa esta celda
# `split(',')` devuelve texto siempre: si no conviertes, estarás sumando cadenas.
# Para acumular por producto usa un diccionario con `.get(clave, 0)`, no una lista.

# Apartado 1
def procesar_linea(linea_texto):
    try:   
        texto_separado = linea_texto.split(',')
        #print(texto_separado)
        #print(texto_separado)
        dict_texto = {}
        dict_texto["producto"] = texto_separado[0]
        dict_texto["cantidad"] = int(texto_separado[1])
        dict_texto["precio"] = float(texto_separado[2])
        #print(dict_texto)
        return dict_texto
    except:
        return None


# Apartado 2
ventas_procesadas = []
descartadas = 0
print(datos_ventas_texto.strip().splitlines())
for l in datos_ventas_texto.strip().splitlines():
    resultado = procesar_linea(l)

    if resultado is not None:
        ventas_procesadas.append(resultado)
    else:
        descartadas += 1
        print(f"descartada la venta {l!r}")
        
print("--- Ventas procesadas correctamente ---")
for venta in ventas_procesadas:
    print(venta)

print(f"\nLíneas procesadas con éxito: {len(ventas_procesadas)}")
print(f"Líneas descartadas (errores): {descartadas}")

# Apartado 3
quantity_ventas = 0
price_ventas = 0
for v in ventas_procesadas:
    quantity_ventas += v.get('cantidad')
    price_ventas += v.get('precio')

print(f"Ingresos totales = {(quantity_ventas * price_ventas):.2f}")
print(f"El producto con más unidades: {max(ventas_procesadas, key=lambda x: x['cantidad'])}")




['Laptop,2,1250.50', 'Mouse,5,25.00', 'Teclado,3,75.99', 'Monitor,ERROR,450.00', 'Laptop,1,1200.00', 'Webcam,4,50.25', 'Mouse,10,22.50', 'Teclado,1,75.99']
descartada la venta 'Monitor,ERROR,450.00'
--- Ventas procesadas correctamente ---
{'producto': 'Laptop', 'cantidad': 2, 'precio': 1250.5}
{'producto': 'Mouse', 'cantidad': 5, 'precio': 25.0}
{'producto': 'Teclado', 'cantidad': 3, 'precio': 75.99}
{'producto': 'Laptop', 'cantidad': 1, 'precio': 1200.0}
{'producto': 'Webcam', 'cantidad': 4, 'precio': 50.25}
{'producto': 'Mouse', 'cantidad': 10, 'precio': 22.5}
{'producto': 'Teclado', 'cantidad': 1, 'precio': 75.99}

Líneas procesadas con éxito: 7
Líneas descartadas (errores): 1
Ingresos totales = 70205.98
El producto con más unidades: {'producto': 'Mouse', 'cantidad': 10, 'precio': 22.5}


### Mis reflexiones

*(Escribe aquí tu respuesta antes de seguir)*

- El dueño te pidió "su producto estrella". Calcula también los **ingresos** por producto.
  ¿Sigue siendo la misma respuesta? ¿Cuál de las dos le darías tú, y qué le dirías al
  entregársela?
- Has descartado una línea. ¿Qué información se ha ido con ella y cómo lo reflejarías en el
  informe?
- Usaste un diccionario para acumular las unidades. ¿Cómo lo habrías hecho con una lista, y
  por qué es peor?

---
## Preguntas de recap – Estructuras, funciones y errores

1. Tengo las ventas del día y necesito saber cuánto lleva acumulado cada producto.
   ¿Lista o diccionario? ¿Por qué?
2. ¿Qué diferencia hay entre que una función haga `print` y que haga `return`? ¿Cómo te das
   cuenta de que te has equivocado?
3. ¿Por qué `except ValueError:` es mejor que `except:` a secas?
4. Si `split(',')` te devuelve `['Laptop', '2', '1250.50']`, ¿qué pasa si sumas el segundo
   elemento a un número sin más?
5. ¿Qué hace `.get('clave', 0)` que no haga `['clave']`?

## 4. Comprensiones de listas

Un patrón aparece constantemente: crear una lista vacía, recorrer otra cosa, y ir añadiendo
elementos que cumplen algo o que se han transformado de alguna forma.

```python
caros = []
for p in productos:
    if p['precio'] > 100:
        caros.append(p['nombre'])
```

Python tiene una forma de escribir eso en una línea, la **comprensión de listas**:

```python
caros = [p['nombre'] for p in productos if p['precio'] > 100]
```

Se lee de izquierda a derecha como una frase: *"el nombre, de cada producto, si su precio
pasa de 100"*. Las tres partes son siempre las mismas —qué me llevo, de dónde, con qué
condición— y la condición es opcional.

No es solo más corto: es **más difícil de romper**, porque no hay una lista vacía que se te
olvide inicializar ni un `append` que puedas poner en el sitio equivocado.

**Cuándo se usa:** cuando construyes una lista a partir de otra cosa filtrando o
transformando.
**Cuándo NO:** cuando dentro del bucle pasan varias cosas, hay `if/else` anidados o el
resultado no cabe cómodamente en una línea. Una comprensión de tres líneas con dos
condiciones es peor que el bucle que sustituye. La claridad manda sobre la brevedad.

### Demostración guiada — el bucle y su comprensión

El mismo resultado escrito de las dos formas, y un caso donde la comprensión es mala idea.

In [ ]:
productos = [
    {'nombre': 'Laptop', 'precio': 1250.50, 'stock': 3},
    {'nombre': 'Mouse', 'precio': 25.00, 'stock': 40},
    {'nombre': 'Teclado', 'precio': 75.99, 'stock': 2},
    {'nombre': 'Monitor', 'precio': 450.00, 'stock': 12},
]

# --- 1. El bucle de toda la vida ----------------------------------------
caros_bucle = []
for p in productos:
    if p['precio'] > 100:
        caros_bucle.append(p['nombre'])
print("Con bucle      :", caros_bucle)

# --- 2. La misma idea, una línea ----------------------------------------
caros = [p['nombre'] for p in productos if p['precio'] > 100]
print("Con comprensión:", caros)

# --- 3. Transformar, no solo filtrar -------------------------------------
# Sin condición: se aplica a todos
con_iva = [round(p['precio'] * 1.21, 2) for p in productos]
print("\nPrecios con IVA:", con_iva)

# El valor inmovilizado en cada producto
valor_stock = [p['precio'] * p['stock'] for p in productos]
print(f"Valor total del almacén: {sum(valor_stock):,.2f} €")

# --- 4. Cuándo NO usarla -------------------------------------------------
# Esto es legal y es ilegible. No lo escribas.
etiquetas_mal = [('AGOTADO' if p['stock'] == 0 else 'POCO' if p['stock'] < 5 else 'OK')
                 for p in productos]

# Con un bucle se entiende de un vistazo, y de eso se trata
etiquetas = []
for p in productos:
    if p['stock'] == 0:
        etiquetas.append('AGOTADO')
    elif p['stock'] < 5:
        etiquetas.append('POCO')
    else:
        etiquetas.append('OK')

print("\nEtiquetas de stock:", etiquetas)

**Qué se ve.** Los puntos 1 y 2 dan exactamente lo mismo: `['Laptop', 'Monitor']`. Cuatro
líneas contra una, y en la de una no hay forma de olvidarse de inicializar la lista.

El punto 3 enseña la otra mitad del asunto: una comprensión no solo filtra, también
**transforma**. `[p['precio'] * 1.21 for p in productos]` devuelve una lista nueva con todos
los elementos cambiados. Si esto te recuerda a la vectorización de NumPy que viene en la
sesión 02, es porque es la misma idea; allí además será mucho más rápida.

Y el punto 4 es el aviso. Las dos versiones dan el mismo resultado y la primera cabe en dos
líneas, pero hay que leerla tres veces para saber qué hace. **Una comprensión que no se
entiende de una pasada ha dejado de ser una mejora.**

## 5. Clases: cuando los datos traen reglas puestas

Hasta ahora un producto ha sido un diccionario: `{'nombre': 'Laptop', 'precio': 1250.5,
'stock': 3}`. Funciona perfectamente, y para muchas cosas es todo lo que necesitas.

El problema aparece cuando ese producto tiene **reglas propias**. El stock no puede quedar
negativo. El valor del inventario es precio por stock. Al mostrarlo por pantalla se enseña
siempre con el mismo formato. Con diccionarios, esas reglas viven sueltas por el programa: la
comprobación de que el stock no baje de cero hay que escribirla en cada sitio donde se
descuente stock, y el día que te olvides en uno, tendrás un almacén con −4 unidades.

Una **clase** es la forma de guardar los datos y sus reglas en el mismo sitio:

```python
class Producto:
    def __init__(self, nombre, precio, stock):   # se ejecuta al crear el objeto
        self.nombre = nombre                     # self es "este objeto en concreto"
        ...

    def actualizar_stock(self, cantidad):        # un método: una regla del objeto
        ...
```

Tres piezas y ya está: **`__init__`** construye el objeto y le pone sus datos; **`self`** es
la forma que tiene el objeto de referirse a sí mismo, y va como primer parámetro de todos los
métodos; y **`__str__`** es el método especial que Python llama cuando haces `print(objeto)`,
y sin el cual verás algo como `<__main__.Producto object at 0x7f...>`.

**Cuándo escribir una clase:** cuando los mismos datos van siempre acompañados de las mismas
operaciones, o cuando te descubres repitiendo una comprobación en varios sitios.
**Cuándo NO:** cuando solo necesitas transportar unos campos de un lado a otro. Un
diccionario es más ligero y más rápido de escribir, y no pasa nada por usarlo.
**Qué asume:** que la regla vale para todos los objetos de esa clase. Si cada caso tiene su
propia excepción, la clase acaba llena de `if` y estorba más de lo que ayuda.

### Demostración guiada — una clase con una regla dentro

Una cuenta bancaria: los datos son el titular y el saldo, y la regla es que no se puede
sacar más de lo que hay. Fíjate en que la regla está **dentro** del objeto.

In [ ]:
class CuentaBancaria:
    def __init__(self, titular, saldo_inicial=0):
        self.titular = titular
        self.saldo = saldo_inicial
        self.movimientos = []

    def ingresar(self, cantidad):
        self.saldo += cantidad
        self.movimientos.append(cantidad)

    def retirar(self, cantidad):
        """Saca dinero. La regla: nunca se queda en negativo."""
        if cantidad > self.saldo:
            print(f"   [rechazado] {self.titular} quiso sacar {cantidad} € "
                  f"y solo tiene {self.saldo} €")
            return False
        self.saldo -= cantidad
        self.movimientos.append(-cantidad)
        return True

    def __str__(self):
        return f"Cuenta de {self.titular} | Saldo: {self.saldo:.2f} € | {len(self.movimientos)} movimientos"


# Crear objetos: cada uno con sus propios datos
cuenta_ana = CuentaBancaria("Ana Torres", 500)
cuenta_luis = CuentaBancaria("Luis Gómez")

cuenta_ana.ingresar(250)
cuenta_ana.retirar(100)
cuenta_ana.retirar(5000)          # la regla lo impide
cuenta_luis.ingresar(80)

print(cuenta_ana)                  # print() llama a __str__
print(cuenta_luis)

# Son objetos independientes: tocar uno no afecta al otro
print(f"\nSaldo de Ana: {cuenta_ana.saldo} | Saldo de Luis: {cuenta_luis.saldo}")

# Y se guardan en listas o diccionarios como cualquier otra cosa
banco = [cuenta_ana, cuenta_luis]
print(f"Dinero total en el banco: {sum(c.saldo for c in banco):.2f} €")

**Qué se ve.** El intento de sacar 5.000 € se rechaza, y lo importante es **dónde** se
rechaza: dentro de `retirar()`. Nadie que use esta clase puede saltarse la regla, ni
queriendo ni por descuido, porque no hay otra forma de tocar el saldo. Con un diccionario,
esa comprobación habría que repetirla en cada sitio del programa donde se saque dinero.

Tres cosas más que conviene señalar:

- **`saldo_inicial=0`** es un parámetro con valor por defecto: la cuenta de Luis se creó sin
  indicarlo. Ahorra escribir lo obvio.
- **Cada objeto tiene sus propios datos.** `cuenta_ana.saldo` y `cuenta_luis.saldo` son
  variables distintas aunque salgan de la misma clase. Eso es lo que significa `self`.
- **`__str__` es lo que hace que `print(objeto)` sea legible.** Sin él, Python imprime la
  dirección de memoria, que no le dice nada a nadie.

Y la última línea enseña que los objetos no son un mundo aparte: se meten en listas, se
recorren y se suman como cualquier otro valor.

---
## Ejercicio 2 — El inventario de la tienda

**Contexto.** Sigues en la tienda de informática. El dueño ha visto tu informe de ventas y
ahora quiere algo más ambicioso: llevar el inventario en condiciones. Hasta hoy lo lleva en
una hoja de cálculo donde, dice él, "a veces salen números raros" — que traducido es que
tiene productos con stock negativo porque alguien registró una venta de unidades que no
había. Te pide que montes la base de algo que no permita eso.

**Tu tarea.**

1. **Crea una clase `Producto`:**
   - El constructor (`__init__`) recibe `nombre`, `precio` y `stock_inicial`.
   - Un método `actualizar_stock(cantidad)` que sume o reste del stock actual. **El stock
     nunca puede quedar negativo.**
   - Un método `valor_inventario()` que devuelva el valor de las unidades en stock
     (`precio * stock`).
   - El método especial `__str__`, para que al imprimir un producto se vea un resumen
     legible, del estilo `"Producto: Laptop | Precio: $1250.50 | Stock: 10 unidades"`.
2. **Gestiona un inventario:**
   - Crea una lista `inventario` con al menos **3** productos distintos.
   - Recorre la lista imprimiendo cada producto.
   - Usa una **comprensión de listas** para crear `productos_poco_stock` con los **nombres**
     de los productos cuyo stock sea inferior a 5 unidades.
3. **Pruébalo con el caso que le preocupa al dueño:** intenta descontar más unidades de las
   que hay y comprueba que tu clase no lo permite.

**Criterio de que lo has hecho bien:** después de intentar vender más unidades de las que
tienes, el stock de ese producto tiene que seguir siendo un número que puedas defender
delante del dueño. Y quien use tu clase debería enterarse de que la operación no se hizo: un
rechazo silencioso es casi tan malo como el stock negativo.

In [ ]:
# TODO: completa esta celda
# Calcula el stock resultante ANTES de asignarlo: es la única forma de rechazar la operación sin haberla hecho ya.
# El método debería decirle a quien lo llama si la operación se hizo o no.


### Mis reflexiones

*(Escribe aquí tu respuesta antes de seguir)*

- Tu clase rechaza sacar más stock del que hay. ¿Qué debería pasar en la tienda de verdad
  cuando un cliente pide 10 unidades y hay 3? Enumera al menos dos comportamientos posibles
  y di cuál implementarías.
- Este mismo inventario se podría haber hecho con una lista de diccionarios. ¿Qué has ganado
  exactamente al usar una clase? Sé concreto: no vale "es más elegante".
- ¿Qué tendría que cambiar en el enunciado para que una clase fuera *mala* idea aquí?

---
## Preguntas de recap – Comprensiones y objetos

1. Escribe con una comprensión de listas: los nombres de los productos cuyo stock sea 0.
2. ¿En qué caso es preferible el bucle normal a la comprensión?
3. ¿Para qué sirve `self` y por qué aparece en todos los métodos?
4. ¿Qué pasa si imprimes un objeto de una clase que no tiene `__str__`?
5. Tienes unos datos que solo vas a leer y pasar de una función a otra. ¿Clase o diccionario?

---
## Ejercicio 3 — El control horario

**Contexto.** Cambias de cliente. Una empresa de treinta personas tiene que llevar registro
de jornada —en España es obligatorio— y lo hace con un lector de tarjeta en la puerta que
escupe un fichero de texto: `nombre,día,hora_entrada,hora_salida`, una línea por jornada. El
lector es antiguo y falla de vez en cuando.

La responsable de RRHH te pide un informe semanal: horas trabajadas por persona y, sobre
todo, **quién ha pasado de 40 horas**, porque esas horas hay que compensarlas y ahora mismo
se está enterando tarde.

Este es el ejercicio que integra toda la sesión: funciones, diccionarios, `try-except`,
comprensiones y una clase.

**Tu tarea.**

1. **Una función `procesar_fichaje(linea)`** que devuelva un diccionario con el nombre, el
   día y los **minutos trabajados**, o `None` si la línea no es utilizable. Ojo: hay más de
   una forma de que una línea esté mal.
2. **Una clase `Empleado`** que guarde el nombre y las jornadas de esa persona, con un
   método para añadir una jornada, otro que devuelva las **horas totales** y un `__str__`
   legible.
3. **Recorre el registro** y construye un **diccionario** que vaya del nombre a su objeto
   `Empleado`. Crea el empleado la primera vez que aparece.
4. **Una comprensión de listas** con los nombres de quienes pasan de 40 horas.
5. **El informe**: las horas de cada persona, quién pasa de 40, y cuántas líneas se han
   descartado.

**Criterio de que lo has hecho bien:** tienen que descartarse líneas —si no descartas
ninguna, tu función no está detectando los dos tipos de fallo—. Y antes de dar el informe por
bueno, mira **de quién** son las líneas que has tirado. Si no se reparten por igual entre las
tres personas, tienes un problema que contar en la reunión.

In [ ]:
# El volcado semanal del lector de tarjetas.
registro_fichajes = """
Ana Torres,2026-09-14,09:00,17:30
Luis Gómez,2026-09-14,08:45,18:00
Ana Torres,2026-09-15,09:10,17:00
Marta Ruiz,2026-09-15,10:00
Luis Gómez,2026-09-15,08:50,19:15
Ana Torres,2026-09-16,09:00,18:45
Marta Ruiz,2026-09-16,09:30,17:30
Luis Gómez,2026-09-16,08:40,20:00
Ana Torres,2026-09-17,09:05,17:20
Marta Ruiz,2026-09-17,09:00,MEDIODIA
Luis Gómez,2026-09-17,08:55,19:30
Ana Torres,2026-09-18,08:55,17:10
Marta Ruiz,2026-09-18,09:15,17:45
Luis Gómez,2026-09-18,09:00,18:30
"""

In [ ]:
# TODO: completa esta celda
# Hay dos formas distintas de que una línea esté mal, y las dos levantan la misma excepción: no te quedes en la primera que encuentres.
# Para agrupar por persona, comprueba si el nombre ya está en el diccionario antes de crear el objeto.


### Mis reflexiones

*(Escribe aquí tu respuesta)*

- Mira el informe y después mira de quién son las líneas descartadas. ¿Entregarías este
  informe a RRHH tal como está? Si no, ¿qué le añadirías antes de mandarlo?
- Has usado un diccionario (`nombre -> Empleado`) y dentro de cada empleado una lista de
  jornadas. ¿Por qué un diccionario por fuera y una lista por dentro, y no al revés?
- Has escrito tres funciones, una clase y una comprensión. Si tuvieras que explicarle a
  alguien **en una frase** cuándo usar cada una de esas herramientas, ¿qué le dirías? Esa
  frase es tu resumen de la sesión.

---
## Cierre de la sesión

Hoy no has usado ninguna librería. Todo lo que has escrito —diccionarios, funciones,
`try-except`, comprensiones, una clase— es Python a secas, y es el suelo sobre el que se
apoya el resto del curso.

Dos ideas conviene que sobrevivan a esta sesión, porque van a volver:

1. **La estructura de datos se elige por cómo vas a buscar**, no por cómo vas a guardar. En
   la sesión 02 aparece el `DataFrame` de pandas, que no es más que la lista de diccionarios
   de hoy con herramientas encima.
2. **Descartar un dato malo es una decisión, y tiene consecuencias.** Hoy ha costado que una
   persona parezca trabajar la mitad que sus compañeras. En la sesión 02 faltará el 20 % de
   las edades del Titanic y habrá que decidir otra vez, con más datos en juego.